# Order Reviews - Silver Transformation

## Parameters

In [0]:
dbutils.widgets.text(
    name="environment",
    defaultValue="dev",
    label="Environment"
)

environment = dbutils.widgets.get("environment").strip().lower()

if environment not in ("dev", "test", "prod"):
    raise ValueError(
        f"Unsupported environment: {environment}. Expected dev, test, or prod."
    )

## Setup

In [0]:
from pyspark.sql.functions import col, trim

catalog = f"ecommerce_{environment}"
source_table = f"{catalog}.bronze.olist_order_reviews"
target_table = f"{catalog}.silver.olist_order_reviews"

## Read Bronze data

In [0]:
bronze_df = spark.table(source_table)

## Transform to Silver

In [0]:
silver_df = (
    bronze_df
    .withColumn(
        "review_comment_title",
        trim(col("review_comment_title"))
    )
    .withColumn(
        "review_comment_message",
        trim(col("review_comment_message"))
    )
)

In [0]:
silver_df = silver_df.replace(
    "",
    None,
    subset=["review_comment_title", "review_comment_message"]
)

## Write to Silver

In [0]:
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)